# Frugal Tonal ASR for Fongbe — starter notebook

This notebook takes you from the data to a valid submission file, step by step:

1. install the libraries;
2. download and check the data;
3. look at the data (volumes, voices, a few recordings);
4. prepare the tonal text targets;
5. fine-tune a baseline model (`facebook/w2v-bert-2.0`, CTC on characters, tones as separate symbols);
6. score it on held-out voices with the official scripts;
7. transcribe the test, pass it through `syllabify()` and check the submission file.

**Runtime.** Written for a free GPU (Kaggle or Google Colab, NVIDIA T4, 16 GB). With the default settings (one audio
archive of about 18 h, 2,000 training steps) the whole notebook took **2 h 32 on a Kaggle T4** (training: 2 h 13).
Change the settings in the next cell to train longer or on more data.

**Expected score.** With these settings the baseline reaches a syllable WER of about **21 %**: 20.6 % on the 300
held-out validation utterances, 21.8 % on the public leaderboard and 21.3 % on the private one.

**This is a baseline, not a winning recipe.** It uses a small part of the data and no language model. Ideas to go further
are listed at the end.

**Rules to keep in mind** (details in `conventions.md` and on the competition page):
- the data is reserved for the competition: do not share or upload it;
- pre-trained models must be under a licence that allows commercial use (`w2v-bert-2.0` is MIT);
- only the Bible text of `Train_religious.csv` is allowed; manual annotation is forbidden;
- prize-winning systems must be frugal: ≤ 1,000 MB, real-time factor ≤ 1 on one CPU core, ≤ 2 GB of memory, offline.

## 0. Settings

In [ ]:
from pathlib import Path

REPO_ID = "JaimeMaLangue-Zindi/JML-Hackathon-fon-asr-data"   # competition data on the Hugging Face Hub

# Where to put the data (about 5 GB with the default settings) and the outputs
if Path("/kaggle/working").exists():                   # Kaggle
    DATA_DIR, OUT_DIR = Path("/tmp/fongbe"), Path("/kaggle/working")
elif Path("/content").exists():                        # Google Colab
    DATA_DIR, OUT_DIR = Path("/content/fongbe"), Path("/content/outputs")
else:                                                  # your own machine
    DATA_DIR, OUT_DIR = Path("fongbe_data"), Path("outputs")
DATA_DIR.mkdir(parents=True, exist_ok=True); OUT_DIR.mkdir(parents=True, exist_ok=True)

# Training data used by this baseline: 1 of the 6 archives of Train_reliable (about 18 h).
# Add archives ("audio_train_reliable_001.tar", ...) to train on more data.
TRAIN_ARCHIVES = ["audio_train_reliable_000.tar"]
TEST_ARCHIVES = ["audio_test_000.tar", "audio_test_001.tar"]

BASE_MODEL = "facebook/w2v-bert-2.0"
MAX_STEPS = 2000          # optimiser steps
BATCH_SIZE = 4            # utterances per forward pass (T4: keep 4)
GRAD_ACCUM = 4            # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LEARNING_RATE = 5e-5
WARMUP_STEPS = 300
MAX_SECONDS = 15.0        # longer training utterances are skipped (memory); 99 % are shorter
N_VALIDATION = 300        # held-out utterances scored after training
SEED = 2026

## 1. Install the libraries

In [ ]:
!pip install -q "transformers==5.14.1" soundfile "rapidfuzz==3.14.5"

## 2. Download and check the data

If the dataset asks you to log in, create a Hugging Face access token (read) and either store it as a Kaggle secret named
`HF_TOKEN` (Add-ons → Secrets) or paste it in the login box that appears below. Never write a token in the notebook.

In [ ]:
import os, hashlib, tarfile
from huggingface_hub import hf_hub_download, login

def try_login():
    if os.environ.get("HF_TOKEN"):
        return
    try:                                   # Kaggle secret
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        return
    except Exception:
        pass
    try:                                   # Colab secret
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        return
    except Exception:
        pass
    print("No HF_TOKEN secret found: if the download fails, run `login()` and paste a read token.")

try_login()

def get(filename):
    return Path(hf_hub_download(REPO_ID, filename, repo_type="dataset", local_dir=DATA_DIR))

SMALL_FILES = ["Train_reliable.csv", "Train_uncertain.csv", "Train_religious.csv", "Test.csv",
               "SampleSubmission.csv", "SHA256SUMS", "conventions.md", "syllabify.py", "normalize.py",
               "score.py", "test_score.py", "measure_frugality.py", "requirements.txt"]
for f in SMALL_FILES:
    get(f)

expected = {name.lstrip("*"): h for h, name in (l.split() for l in (DATA_DIR / "SHA256SUMS").read_text().splitlines() if l.strip())}

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def download_and_extract(archive):
    marker = DATA_DIR / (archive + ".done")
    if marker.exists():
        print("already extracted:", archive); return
    path = get(archive)
    assert sha256(path) == expected[archive], f"corrupted download: {archive} (delete it and run again)"
    with tarfile.open(path) as tar:
        try:
            tar.extractall(DATA_DIR, filter="data")
        except TypeError:                  # Python < 3.12
            tar.extractall(DATA_DIR)
    path.unlink()                          # free the disk: the WAV files are kept
    marker.touch()
    print("ok:", archive)

for archive in TRAIN_ARCHIVES + TEST_ARCHIVES:
    download_and_extract(archive)

## 3. Look at the data

In [ ]:
import sys
import pandas as pd
sys.path.insert(0, str(DATA_DIR))          # official scripts: normalize.py, syllabify.py, score.py
from normalize import tonal_target
from syllabify import syllabify

reliable = pd.read_csv(DATA_DIR / "Train_reliable.csv")
uncertain = pd.read_csv(DATA_DIR / "Train_uncertain.csv")
religious = pd.read_csv(DATA_DIR / "Train_religious.csv")
test = pd.read_csv(DATA_DIR / "Test.csv")
sample = pd.read_csv(DATA_DIR / "SampleSubmission.csv")

for name, df in [("Train_reliable", reliable), ("Train_uncertain", uncertain), ("Train_religious", religious)]:
    print(f"{name:16s} {len(df):7,d} utterances  {df.duration.sum() / 3600:6.1f} h  {df.speaker_id.nunique():4d} voices")
print(f"{'Test':16s} {len(test):7,d} utterances (audio only)")
reliable.head()

In [ ]:
from IPython.display import Audio, display

for _, row in reliable.sample(2, random_state=SEED).iterrows():
    wav = DATA_DIR / row.audio_path
    if wav.exists():
        print(row.id, row.speaker_id, f"{row.duration:.1f} s:", row.transcription)
        display(Audio(str(wav)))

## 4. Text targets

The score only looks at letters and tones. `tonal_target()` from the official `normalize.py` gives exactly that: Unicode
NFD, lower case, punctuation and digits removed. In NFD each tone is a separate combining character, so the model learns
the tones as symbols of their own. `syllabify()` then shows what the leaderboard compares.

In [ ]:
example = reliable.transcription.iloc[1]
print("original   :", example)
print("target     :", tonal_target(example))
print("syllabified:", syllabify(example))

In [ ]:
import json, random
import numpy as np
from transformers import Wav2Vec2CTCTokenizer

random.seed(SEED); np.random.seed(SEED)

reliable["target"] = reliable.transcription.map(tonal_target).str.strip()
reliable["wav"] = reliable.audio_path.map(lambda p: str(DATA_DIR / p))
available = reliable[reliable.wav.map(lambda p: Path(p).exists()) & (reliable.target != "")]

# Voices marked recommended_validation = 1 are never used for training (unseen voices, as in the test)
train_df = available[(available.recommended_validation == 0) & (available.duration <= MAX_SECONDS)]
valid_df = available[available.recommended_validation == 1]
valid_df = valid_df.sample(min(N_VALIDATION, len(valid_df)), random_state=SEED)
print(f"train: {len(train_df):,} utterances, {train_df.duration.sum() / 3600:.1f} h, {train_df.speaker_id.nunique()} voices")
print(f"validation: {len(valid_df):,} utterances, {valid_df.speaker_id.nunique()} voices (not in training)")

chars = sorted(set("".join(train_df.target)) - {" "})
vocab = {c: i for i, c in enumerate(chars)}
vocab["|"] = len(vocab); vocab["[UNK]"] = len(vocab); vocab["[PAD]"] = len(vocab)
(OUT_DIR / "vocab.json").write_text(json.dumps(vocab, ensure_ascii=False))
tokenizer = Wav2Vec2CTCTokenizer(str(OUT_DIR / "vocab.json"), unk_token="[UNK]", pad_token="[PAD]",
                                 word_delimiter_token="|", bos_token=None, eos_token=None)   # no <s> </s> symbols
print(len(vocab), "symbols:", " ".join(chars))

## 5. Fine-tune the baseline

`facebook/w2v-bert-2.0` (600 M parameters, MIT licence) with a CTC head on the symbols above. The recipe follows the
Hugging Face guide for low-resource ASR with this encoder. Batches group utterances of similar length to save memory.

In [ ]:
import math, time
import soundfile as sf
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (SeamlessM4TFeatureExtractor, Wav2Vec2BertProcessor, Wav2Vec2BertForCTC,
                          get_linear_schedule_with_warmup)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
torch.manual_seed(SEED)

feature_extractor = SeamlessM4TFeatureExtractor.from_pretrained(BASE_MODEL)
processor = Wav2Vec2BertProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

def load_audio(path):
    audio, sr = sf.read(path, dtype="float32")
    assert sr == 16000, path
    return audio if audio.ndim == 1 else audio.mean(axis=1)

class SpeechSet(Dataset):
    def __init__(self, df, with_labels=True):
        self.paths = df.wav.tolist()
        self.texts = df.target.tolist() if with_labels else None
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        item = {"input_features": feature_extractor(load_audio(self.paths[i]), sampling_rate=16000).input_features[0]}
        if self.texts is not None:
            item["labels"] = tokenizer(self.texts[i]).input_ids
        return item

def collate(items):
    batch = processor.pad([{"input_features": x["input_features"]} for x in items], padding=True, return_tensors="pt")
    if "labels" in items[0]:
        labels = processor.pad(labels=[{"input_ids": x["labels"]} for x in items], padding=True, return_tensors="pt")
        batch["labels"] = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
    return batch

def length_batches(durations, batch_size, shuffle):
    """Batches of utterances of similar length (less padding); random order of the batches when training."""
    order = np.argsort(durations)
    batches = [order[i:i + batch_size].tolist() for i in range(0, len(order), batch_size)]
    if shuffle:
        random.shuffle(batches)
    return batches

model = Wav2Vec2BertForCTC.from_pretrained(
    BASE_MODEL, attention_dropout=0.0, hidden_dropout=0.0, feat_proj_dropout=0.0, mask_time_prob=0.0,
    layerdrop=0.0, ctc_loss_reduction="mean", ctc_zero_infinity=True, add_adapter=True,
    pad_token_id=tokenizer.pad_token_id, vocab_size=len(tokenizer)).to(device)
model.gradient_checkpointing_enable()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f} M parameters")

In [ ]:
train_set = SpeechSet(train_df)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0)
scheduler = get_linear_schedule_with_warmup(optimizer, WARMUP_STEPS, MAX_STEPS)
scaler = torch.cuda.amp.GradScaler(enabled=device == "cuda")

model.train()
step, micro, running, start = 0, 0, [], time.time()
while step < MAX_STEPS:
    loader = DataLoader(train_set, batch_sampler=length_batches(train_df.duration.values, BATCH_SIZE, True),
                        collate_fn=collate, num_workers=2)
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
            loss = model(**batch).loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        running.append(loss.item() * GRAD_ACCUM)
        micro += 1
        if micro % GRAD_ACCUM:
            continue
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()
        step += 1
        if step % 100 == 0:
            print(f"step {step:5d}/{MAX_STEPS}  loss {np.mean(running):.3f}  {(time.time() - start) / 60:5.1f} min")
            running = []
        if step >= MAX_STEPS:
            break

model.save_pretrained(OUT_DIR / "model"); processor.save_pretrained(OUT_DIR / "model")
print("saved in", OUT_DIR / "model")

## 6. Score on held-out voices

Greedy CTC decoding, then the official `score.py`. Its `TSER` is the leaderboard metric (WER on syllables); the other
values are tonal diagnostics that do not count for the ranking.

In [ ]:
import subprocess

@torch.no_grad()
def transcribe(df, batch_size=8):
    model.eval()
    data = SpeechSet(df, with_labels=False)
    batches = length_batches(df.duration.values if "duration" in df else np.zeros(len(df)), batch_size, False)
    out = [""] * len(df)
    for idx in batches:
        batch = collate([data[i] for i in idx])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
            logits = model(input_features=batch["input_features"].to(device),
                           attention_mask=batch["attention_mask"].to(device)).logits
        for i, text in zip(idx, processor.batch_decode(logits.argmax(-1).cpu(), skip_special_tokens=True)):
            out[i] = text
    return out

valid_df = valid_df.copy()
valid_df["prediction"] = transcribe(valid_df)
valid_df[["id", "transcription"]].to_csv(OUT_DIR / "valid_reference.csv", index=False)
valid_df[["id", "prediction"]].rename(columns={"prediction": "transcription"}).to_csv(OUT_DIR / "valid_prediction.csv", index=False)
print(subprocess.run([sys.executable, str(DATA_DIR / "score.py"), "--reference", str(OUT_DIR / "valid_reference.csv"),
                      "--submission", str(OUT_DIR / "valid_prediction.csv")], capture_output=True, text=True).stdout)
for _, r in valid_df.head(5).iterrows():
    print("REF:", r.transcription, "\nHYP:", r.prediction, "\n")

## 7. Transcribe the test and build the submission

In [ ]:
test = pd.read_csv(DATA_DIR / "Test.csv")
test["wav"] = test.audio_path.map(lambda p: str(DATA_DIR / p))
missing = (~test.wav.map(lambda p: Path(p).exists())).sum()
assert missing == 0, f"{missing} test files missing: check TEST_ARCHIVES"
test["duration"] = test.wav.map(lambda p: sf.info(p).duration)

test["transcription"] = transcribe(test)
test[["id", "transcription"]].to_csv(OUT_DIR / "test_plain_transcriptions.csv", index=False)   # keep for your analyses

# Zindi submission format: columns ID, Target (see SampleSubmission.csv)
submission = pd.DataFrame({"ID": test["id"], "Target": test.transcription.fillna("").map(syllabify)})   # MANDATORY
submission.to_csv(OUT_DIR / "submission.csv", index=False)

In [ ]:
# Checks before uploading submission.csv to Zindi
sub = pd.read_csv(OUT_DIR / "submission.csv", keep_default_na=False)
assert list(sub.columns) == ["ID", "Target"], sub.columns
assert sub.ID.tolist() == sample.ID.tolist(), "IDs or order differ from SampleSubmission.csv"
assert sub.ID.is_unique
import re
assert all(re.fullmatch(r"[a-zɔɖɛ1-5 ]*", t) for t in sub.Target), "transcriptions are not syllabified"
print(f"{len(sub):,} rows, {(sub.Target == '').sum()} empty transcriptions")
print(sub.head())
print("Upload", OUT_DIR / "submission.csv", "on the competition page (Submit).")

## 8. Going further

- **More data**: the other archives of `Train_reliable`, the Bible readings (`Train_religious`, one reader) and the
  200 h of `Train_uncertain` (uncertain tones: filter or weight them automatically, e.g. by the confidence of a first model).
- **Longer training** and a learning-rate schedule tuned on the validation voices.
- **A language model** built only from the training transcriptions (for example KenLM + `pyctcdecode`).
- **Frugality**: the final system must run on one CPU core with ≤ 2 GB of memory and weigh ≤ 1,000 MB. This 600 M
  baseline is a starting point; quantisation (int8) or a smaller encoder may be needed. Check it with the official script
  (Linux, root rights to cut the network):

```bash
python measure_frugality.py --weights my_system/ \
    --command "python infer.py --csv {csv} --root {root} --output {output}" --csv Train_reliable.csv --limit 300
```

The next cell writes a minimal CPU `infer.py` for this baseline that you can use with that command.

In [ ]:
INFER = """import argparse, csv, sys
import numpy as np, soundfile as sf, torch
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor

ap = argparse.ArgumentParser()
ap.add_argument("--csv", required=True); ap.add_argument("--root", required=True); ap.add_argument("--output", required=True)
ap.add_argument("--model", default="model")
a = ap.parse_args()
torch.set_num_threads(1)
processor = Wav2Vec2BertProcessor.from_pretrained(a.model)
model = Wav2Vec2BertForCTC.from_pretrained(a.model).eval()
model = torch.quantization.quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)
rows = list(csv.DictReader(open(a.csv, encoding="utf-8")))
with open(a.output, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["id", "transcription"])
    for r in rows:
        audio, sr = sf.read(f"{a.root}/{r['audio_path']}", dtype="float32")
        feats = processor(audio, sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            ids = model(**feats).logits.argmax(-1)
        w.writerow([r["id"], processor.batch_decode(ids, skip_special_tokens=True)[0]])
"""
(OUT_DIR / "infer.py").write_text(INFER)
print("written:", OUT_DIR / "infer.py", "(plain transcriptions: apply syllabify() before any submission)")